# IEM Cologne 2026 — CS2 Major Predictor

End-to-end notebook: **collect CS2-only data → engineer features → train the stacked
ensemble → backtest → simulate the Major** (Stage 1/2/3 Swiss + single-elim playoffs)
and predict the champion.

It drives the reusable `iemcs` package end-to-end and shows the tables and charts inline.

> The first run clones the Valve Regional Standings repo into `data/raw/`
> (set the `VRS_REPO_DIR` env var to reuse an existing clone).

In [ ]:
import os, sys

# Find the repo root (the folder containing the `iemcs` package) from any cwd.
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "iemcs")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from iemcs import config, vrs, dataset, tournament, validate, report
from iemcs.model import MatchModel
from iemcs.config import FEATURE_COLUMNS
from iemcs.teams import FIELD

# `iemcs.report` forces the Agg backend on import; restore inline afterwards so the
# notebook's own plots render in the cells.
%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

N_SIMS = 50_000          # Monte-Carlo runs (the simulation is fast; tune freely)
print("iemcs loaded ·", len(FEATURE_COLUMNS), "engineered features ·", len(FIELD), "teams")

## 1 · Collect CS2-only data

The single source is Valve's official Regional Standings repo. Its per-team monthly
*details* files are parsed into deduplicated CS2 matches (no CS:GO), plus VRS points,
rosters and regions.

In [ ]:
vrs.ensure_repo()
matches   = vrs.parse_matches()
standings = vrs.parse_standings()
region    = vrs.build_region_map()

print(f"{len(matches):,} CS2 matches   {matches.date.min().date()} -> {matches.date.max().date()}")
print(f"{standings.snapshot.nunique()} monthly VRS snapshots · {len(region)} teams with a region")
matches.head()

In [ ]:
# Match volume over time (the training data)
mc = matches.set_index("date").resample("MS").size()
labels = [d.strftime("%Y-%m") for d in mc.index]
plt.figure(figsize=(12, 3))
plt.bar(range(len(mc)), mc.values, color="#2c7fb8")
plt.xticks(range(len(mc)), labels, rotation=90, fontsize=7)
plt.title("CS2 professional matches per month"); plt.ylabel("matches")
plt.tight_layout(); plt.show()

## 2 · Feature engineering (leak-free)

One chronological pass evolves each team's Elo, Glicko-2, form, streak, activity,
head-to-head and strength-of-schedule, recording *pre-match* features before observing
each result. The fitted `Context` then holds every team's present strength.

In [ ]:
frame, ctx = dataset.build(matches, standings, region)
print("feature frame:", frame.shape, "| target balance:", round(frame.y.mean(), 3))
frame[FEATURE_COLUMNS + ["y"]].head()

In [ ]:
# Current strength of the 32-team field
cur  = vrs.current_standings()
cmap = dict(zip(cur.team, cur.points))
now  = matches.date.max()
ladder = pd.DataFrame([
    {"team": t.name, "stage": t.stage,
     "Elo": round(ctx.elo.rating(t.vrs_name)),
     "Glicko": round(ctx.glicko.rating(t.vrs_name)),
     "RD": round(ctx.glicko.rd(t.vrs_name)),
     "VRS": cmap.get(t.vrs_name, 1000)}
    for t in FIELD
]).sort_values("Elo", ascending=False).reset_index(drop=True)
ladder.head(12)

## 3 · Train the stacked ensemble

Base learners (logistic regression, random forest, hist-gradient-boosting) + the Elo /
Glicko-2 / VRS expectations are combined by a logistic meta-learner on out-of-fold
predictions, isotonically calibrated, with a cross-region VRS blend tuned on held-out
cross-region matches.

In [ ]:
from iemcs.torch_model import _HAS_TORCH
model = MatchModel().fit(frame, verbose=True)
print("PyTorch MLP active:", _HAS_TORCH, "| base learners:", list(model.base))
pd.DataFrame({"member": model.stack_columns,
              "meta_weight": model.meta.coef_[0].round(3)})

## 4 · Backtest & feature importance

Walk-forward (train on the past, predict the next block) vs rating-only baselines, plus
the out-of-sample calibration curve and permutation feature importance.

In [ ]:
bt = validate.backtest(frame, n_splits=3)
validate.print_report(bt)
pd.DataFrame({k: bt[k] for k in ["ensemble", "elo_only", "vrs_only"]}).T[
    ["accuracy", "log_loss", "brier", "auc", "n"]].round(4)

In [ ]:
from sklearn.calibration import calibration_curve
y, p = bt["_reliability"]
frac, mean_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
plt.plot(mean_pred, frac, "o-", color="#c0392b", label="ensemble")
plt.xlabel("predicted probability"); plt.ylabel("observed frequency")
plt.title("Calibration (out-of-sample)"); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
# Permutation importance + standalone AUC for each feature
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

k = int(len(frame) * 0.8)
tr, te = frame.iloc[:k], frame.iloc[k:]
gbm = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_depth=4,
                                     l2_regularization=1.0, random_state=0)
gbm.fit(tr[FEATURE_COLUMNS], tr.y)
pi = permutation_importance(gbm, te[FEATURE_COLUMNS], te.y,
                            scoring="neg_log_loss", n_repeats=8, random_state=0)
imp = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "uni_AUC_gap": [abs(roc_auc_score(te.y, te[f]) - 0.5) for f in FEATURE_COLUMNS],
    "perm_importance": pi.importances_mean,
}).sort_values("perm_importance", ascending=False).reset_index(drop=True)

ax = imp.set_index("feature")["perm_importance"].iloc[::-1].plot.barh(
    figsize=(8, 5), color="#31a354")
ax.set_title("Permutation importance (drop in log-loss when shuffled)")
plt.tight_layout(); plt.show()
imp.round(4)

## 4b · Model comparison — pre vs post ensemble

Out-of-sample (time-held-out last 20%) metrics for **each base learner on its own**, the
rating signals for reference, and the stacked ensemble **without** vs **with** the new
KNN member. Lower `log_loss`/`brier` and higher `auc`/`accuracy` are better.

The KNN member adds instance-based diversity; the `ENSEMBLE (no knn)` vs `ENSEMBLE
(full)` rows isolate its effect. Note that stacking buys *robust, calibrated*
probabilities rather than a guaranteed log-loss win on every split — on a single recent
window the strongest single model can edge it, and the cross-region VRS blend trades a
little raw log-loss for sanity in the *simulation* (it stops weak-region teams being
over-rated in the cross-region matchups that dominate a Major).

In [ ]:
comp = validate.compare_models(frame, new_member="knn")
comp.round(4)

## 4c · End-to-end simulator backtest (real Majors)

The match-level backtest above tests single predictions; this tests the **whole stack**
(ratings → ensemble → series math → bracket) against reality. It replays the actual
8-team playoff brackets of the last two CS2 Majors using a model trained **only on
pre-event data**, and compares predicted reach-SF / reach-final / champion
probabilities to what really happened. (Both were won by Vitality, so the discriminating
signal is the reach-SF / reach-final Brier — did it rank the right four / two teams?)

In [ ]:
from iemcs import simbacktest

sim_bt = simbacktest.run(matches, standings, region, recency_half_life=365, n_sims=20000)
print("End-to-end metrics:", {k: round(v, 4) for k, v in simbacktest.metrics(sim_bt).items()})

majors = list(sim_bt["major"].unique())
for mj in majors:                                    # one table per Major
    print(f"\n=== {mj} ===")
    display(sim_bt[sim_bt.major == mj].drop(columns="major")
            .sort_values("P_champion", ascending=False)
            .reset_index(drop=True).round(3))

In [ ]:
# Predicted title odds per Major (red = the team that actually won)
fig, axes = plt.subplots(1, len(majors), figsize=(13, 4.5))
for ax, mj in zip(axes, majors):
    d = sim_bt[sim_bt.major == mj].sort_values("P_champion")
    colors = ["#c0392b" if a == "Champion" else "#95a5a6" for a in d["actual"]]
    ax.barh(d["team"], d["P_champion"] * 100, color=colors)
    ax.set_title(mj); ax.set_xlabel("predicted P(champion) %"); ax.grid(axis="x", alpha=.3)
fig.suptitle("Simulator backtest — predicted title odds (red = actual champion)")
fig.tight_layout(); plt.show()

## 5 · Simulate the Major

Build the per-map win-probability table for all 32 teams, then Monte-Carlo the full
format (three Swiss stages + single-elim playoffs) `N_SIMS` times.

In [ ]:
vrs_pts = {t.name: float(cmap.get(t.vrs_name, 1000)) for t in FIELD}
pmap = tournament.build_pmap(model, ctx, now)
res  = tournament.run_monte_carlo(pmap, vrs_pts, n_sims=N_SIMS, n_jobs=-1)

assert abs(res.champion.sum() - 1) < 1e-6 and abs(res.adv_s3.sum() - 8) < 1e-6
disp = res[["stage", "vrs", "adv_s1", "adv_s2", "adv_s3",
            "semifinal", "final", "champion"]].copy()
for c in ["adv_s1", "adv_s2", "adv_s3", "semifinal", "final", "champion"]:
    disp[c] = (disp[c] * 100).round(1)
disp

In [ ]:
# Title odds with Monte-Carlo confidence intervals
top = res.sort_values("champion").tail(15)
plt.figure(figsize=(8, 6))
plt.barh(top.index, top.champion * 100,
         xerr=[(top.champion - top.champion_lo) * 100,
               (top.champion_hi - top.champion) * 100],
         color="#c0392b", alpha=.85, error_kw=dict(ecolor="#555", lw=.8))
plt.xlabel("title probability (%)"); plt.title("IEM Cologne 2026 — title odds")
plt.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# Probability of reaching the playoffs (Stage 3 top-8)
po = res.sort_values("adv_s3").tail(20)
plt.figure(figsize=(8, 7))
plt.barh(po.index, po.adv_s3 * 100, color="#2c7fb8", alpha=.85)
plt.xlabel("probability (%)"); plt.title("Reaching the playoffs (top-8 of Stage 3)")
plt.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# Persist the full set of artifacts (CSVs, charts, markdown report)
bt_clean = {k: v for k, v in bt.items() if k != "_reliability"}
csvs   = report.save_csvs(res)
charts = report.charts(res)
md_out = report.write_markdown(res, bt_clean)
print(f"Saved {len(csvs)} CSVs and {len(charts)} charts to {config.OUTPUTS_DIR}")
print("Report:", md_out)

## Notes

- **Vitality** emerge as a heavy favourite because they were dominant across the
  2024–2026 CS2 data the model trains on — it reflects the data, not a hand-set prior.
  Raise `PROB_SHRINK` in `iemcs/config.py` to add upset variance.
- Re-running top-to-bottom closer to the event folds in the latest VRS update and rosters.
- All logic lives in the reusable `iemcs` package; this notebook is the single entry point.